# 01 — Streaming Basics: Examples

> Your first streaming calls — understand chunks, deltas, finish reasons, and metrics.

**What you'll build:**
1. Minimal streaming call
2. Chunk inspector — see every field of every chunk
3. Accumulator pattern — build full response from stream
4. Streaming metrics (TTFT + throughput)
5. Streaming vs. non-streaming comparison

In [ ]:
# !pip install openai python-dotenv rich

In [ ]:
import os, time
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint
from rich.table import Table
from rich.console import Console

load_dotenv()
client = OpenAI()
console = Console()
print('✅ Setup complete')

---
## Example 1: Minimal Streaming Call

In [ ]:
# The simplest possible streaming call
print("Streaming response:\n")

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Count from 1 to 5 with a short pause between each number."}],
    stream=True   # ← One argument enables streaming
)

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)  # Print immediately, don't buffer

print("\n\n✅ Stream complete")

---
## Example 2: Chunk Inspector — What's Inside Each Chunk?

In [ ]:
# Inspect the first 8 chunks in detail
print("🔍 Inspecting stream chunks (first 8 only):\n")

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say 'Hello, streaming world!' word by word."}],
    stream=True
)

table = Table(title="Chunk Inspector", show_header=True)
table.add_column("#", style="cyan", width=4)
table.add_column("delta.role", style="yellow", width=12)
table.add_column("delta.content", style="green", width=20)
table.add_column("delta.tool_calls", style="white", width=16)
table.add_column("finish_reason", style="red", width=15)

all_chunks = list(stream)  # Collect all chunks

for i, chunk in enumerate(all_chunks[:8]):
    delta = chunk.choices[0].delta
    table.add_row(
        str(i),
        repr(delta.role),
        repr(delta.content) if delta.content else "(empty)",
        str(delta.tool_calls) if delta.tool_calls else "None",
        str(chunk.choices[0].finish_reason)
    )

if len(all_chunks) > 8:
    table.add_row("...", "...", f"({len(all_chunks)-9} more chunks)", "...", "...")
    last = all_chunks[-1]
    table.add_row(
        str(len(all_chunks)-1),
        repr(last.choices[0].delta.role),
        repr(last.choices[0].delta.content) if last.choices[0].delta.content else "(empty)",
        "None",
        str(last.choices[0].finish_reason)
    )

console.print(table)
print(f"\nTotal chunks: {len(all_chunks)}")

---
## Example 3: Accumulator Pattern

In [ ]:
def stream_and_accumulate(prompt: str, model: str = "gpt-4o-mini") -> dict:
    """
    Stream a response and return:
    - full_text: the complete response
    - finish_reason: why the stream ended
    - chunk_count: how many chunks arrived
    """
    full_text = ""
    finish_reason = None
    chunk_count = 0

    print(f"\n🟢 Streaming: \"{prompt[:50]}...\"\n")

    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )

    for chunk in stream:
        chunk_count += 1
        delta = chunk.choices[0].delta

        if delta.content:
            full_text += delta.content
            print(delta.content, end="", flush=True)

        if chunk.choices[0].finish_reason:
            finish_reason = chunk.choices[0].finish_reason

    print()  # Newline after stream
    return {"full_text": full_text, "finish_reason": finish_reason, "chunk_count": chunk_count}


result = stream_and_accumulate(
    "Explain in 2 sentences what Server-Sent Events (SSE) are and why LLMs use them."
)

rprint(f"\n[bold]Stream summary:[/bold]")
rprint(f"   Chunks received: {result['chunk_count']}")
rprint(f"   Finish reason:   {result['finish_reason']}")
rprint(f"   Total chars:     {len(result['full_text'])}")

---
## Example 4: Streaming Metrics (TTFT + Throughput)

In [ ]:
def stream_with_metrics(prompt: str, model: str = "gpt-4o-mini") -> dict:
    """Stream and track TTFT, throughput, and total latency."""
    start = time.perf_counter()
    first_token_time = None
    chunks_with_content = 0
    full_text = ""

    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )

    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            if first_token_time is None:
                first_token_time = time.perf_counter()
            chunks_with_content += 1
            full_text += content
            print(content, end="", flush=True)

    end = time.perf_counter()
    print()

    total_ms = (end - start) * 1000
    ttft_ms  = (first_token_time - start) * 1000 if first_token_time else None
    gen_ms   = (end - first_token_time) * 1000 if first_token_time else None
    tps      = chunks_with_content / (gen_ms / 1000) if gen_ms else 0

    return {
        "ttft_ms":     round(ttft_ms, 1) if ttft_ms else None,
        "total_ms":    round(total_ms, 1),
        "gen_ms":      round(gen_ms, 1) if gen_ms else None,
        "tokens":      chunks_with_content,
        "throughput":  round(tps, 1),
        "full_text":   full_text,
    }


# ── Test ───────────────────────────────────────────────────────────────────
prompts = [
    "What is 2 + 2?",
    "Write a 3-sentence explanation of neural networks.",
    "List the 5 most important Python built-in functions with one-sentence descriptions.",
]

metrics_results = []
for p in prompts:
    print(f"\n📤 '{p[:50]}'\n")
    m = stream_with_metrics(p)
    m["prompt"] = p[:40]
    metrics_results.append(m)

# ── Display metrics comparison ─────────────────────────────────────────────
table = Table(title="Streaming Performance Metrics", show_header=True)
table.add_column("Prompt", style="cyan", max_width=38)
table.add_column("TTFT", style="yellow")
table.add_column("Total", style="white")
table.add_column("Tokens", style="green")
table.add_column("Throughput", style="green")

for m in metrics_results:
    table.add_row(
        m["prompt"],
        f"{m['ttft_ms']}ms",
        f"{m['total_ms']}ms",
        str(m["tokens"]),
        f"{m['throughput']} tok/s"
    )

console.print(table)

---
## Example 5: Streaming vs. Non-Streaming Latency Comparison

In [ ]:
def non_streaming_call(prompt: str, model: str = "gpt-4o-mini") -> dict:
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200
    )
    total = (time.perf_counter() - start) * 1000
    return {
        "total_ms": round(total, 1),
        "ttft_ms":  round(total, 1),  # No TTFT advantage — all or nothing
        "tokens":   response.usage.completion_tokens,
    }


def streaming_call(prompt: str, model: str = "gpt-4o-mini") -> dict:
    start = time.perf_counter()
    first = None
    tokens = 0
    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            if first is None: first = time.perf_counter()
            tokens += 1
    total = (time.perf_counter() - start) * 1000
    ttft = (first - start) * 1000 if first else None
    return {
        "total_ms": round(total, 1),
        "ttft_ms":  round(ttft, 1) if ttft else None,
        "tokens":   tokens,
    }


test_prompt = "Write a 3-sentence overview of Python's async/await model."

print("⏱️  Comparing streaming vs non-streaming...\n")
s = streaming_call(test_prompt)
n = non_streaming_call(test_prompt)

table = Table(title="Streaming vs Non-Streaming", show_header=True)
table.add_column("Mode", style="cyan")
table.add_column("Time-To-First-Token", style="yellow")
table.add_column("Total Time", style="white")
table.add_column("Tokens", style="green")

table.add_row("🌊 Streaming",     f"{s['ttft_ms']}ms",       f"{s['total_ms']}ms", str(s['tokens']))
table.add_row("⬛ Non-streaming", f"{n['ttft_ms']}ms (full)", f"{n['total_ms']}ms", str(n['tokens']))

console.print(table)

if s['ttft_ms']:
    speedup = n['ttft_ms'] / s['ttft_ms']
    rprint(f"\n✅ Streaming shows first token [bold green]{speedup:.1f}×[/bold green] faster than waiting for full response")

---
## Summary

| Concept | Key Point |
|---|---|
| `stream=True` | One argument enables streaming |
| `delta.content` | The new text in each chunk — accumulate these |
| `finish_reason` | Why streaming stopped: `stop`, `tool_calls`, `length` |
| TTFT | Time to first token — streaming reduces this from seconds to ~200ms |
| `flush=True` | Required to display streamed output immediately in terminal |

**Next:** `02_openai_streaming` — streaming with tool calls, structured output, and the `with_streaming_response` context manager.